In [1]:
import os
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv()

/Users/Francisco_Reveriano/Documents/MultiModal_Models/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
HF_TOKEN =  os.getenv("HF_TOKEN") # Replace with your token from https://huggingface.co/settings/tokens
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

# Check for MPS (Apple Silicon) availability
if torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS (Metal Performance Shaders)")
else:
    device = "cpu"
    print("WARNING: MPS not available, falling back to CPU (very slow)")


/Users/Francisco_Reveriano/Documents/MultiModal_Models/.venv/lib/python3.13/site-packages/diffusers/models/transformers/transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/Users/Francisco_Reveriano/Documents/MultiModal_Models/.venv/lib/python3.13/site-packages/diffusers/models/transformers/transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)


Using Apple MPS (Metal Performance Shaders)


In [4]:
model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

vae = AutoencoderKLWan.from_pretrained(
    model_id, subfolder="vae", torch_dtype=torch.float32
)

pipe = WanPipeline.from_pretrained(
    model_id, vae=vae, torch_dtype=torch.float16
)

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config, flow_shift=3.0
)

# Plenty of RAM — load everything to GPU for max speed
pipe.to(device)

# These still help with peak memory spikes during VAE decode
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

/Users/Francisco_Reveriano/Documents/MultiModal_Models/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████| 242/242 [00:08<00:00, 27.19it/s, Materializing param=shared.weight]
UMT5EncoderModel LOAD REPORT from: /Users/Francisco_Reveriano/.cache/huggingface/hub/models--Wan-AI--Wan2.1-T2V-1.3B-Diffusers/snapshots/0fad780a534b6463e45facd96134c9f345acfa5b/text_encoder
Key                         | Status  | 
----------------------------+---------+-
encoder.embed_tokens.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading pipeline components...: 100%|██████████| 5/5 [00:12<00:00,  2.59s/it]


In [ ]:
# =============================================================
# Cell 4: Generate a video
# =============================================================
prompt = (
    "Two anthropomorphic cats in comfy boxing gear and bright gloves "
    "fight intensely on a spotlighted stage."
)

output = pipe(
    prompt=prompt,
    num_frames=81,         # ~5 seconds at 16fps (must be 4k+1)
    height=480,
    width=832,
    num_inference_steps=50,
    guidance_scale=6.0,
).frames[0]

# =============================================================
# Cell 5: Save and display
# =============================================================
export_to_video(output, "wan_output.mp4", fps=16)
print("✅ Video saved to wan_output.mp4")

from IPython.display import Video
Video("wan_output.mp4", embed=True)

  0%|          | 0/50 [00:00<?, ?it/s]